In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('/home/domans/dianne-codebase/')
import dianne
import viewer
dianne.setNotebookWidth(100)

In [ ]:
dataPath = './data/'

# Download the demo data from Zenodo and unzip it to the data folder
dianne.downloadZIPFromZenodo(dataPath, 'https://zenodo.org/records/19225051/files/', 'RMS2194.oid0.zip')

# Load the data and prepare the patches for annotation and training the classifier
samples  = ['RMS2194.oid0']
classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)
F = 1
fname = f'features/false-{F}-ctranspath_features.tsv.gz'
patch_size = 8 # number of tiles, in each dimension, to include in a patch (e.g. 8 means 8x8=64 tiles per patch)
ts, mpp, tile_size = dianne.loadSTQParams(dataPath + samples[0], F)
ads, imgs, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne.loadDataAndPreparePatches(samples, dataPath, fname, L=None, ts=ts, mpp=mpp, N=patch_size)
sizes = {s: ads[s].shape[0] for s in samples}

runfn = dianne.makeRunFn(patchCoordinates, ads, samples, qs, ts, mpp, tile_size=tile_size, patch_size=patch_size, PCMA_alpha=0.8, alpha_img=0.5)
savefn = dianne.makeSaveFn(patchCoordinates, ads, samples, qs, ts, mpp, PCMA_alpha=0.8, tile_size=tile_size, patch_size=patch_size, body_overlap=0.25, classifierPaths=classifierPaths)
loadfn = dianne.makeLoadFn(classifierPaths)
listfn = dianne.makeListFn(classifierPaths)

drawings = viewer.create_viewer(samples, imgs, height="800px", run_inference_fn=runfn, sample_sizes=sizes,
                               save_func=savefn, load_func=loadfn, list_names_func=listfn)[1]

In [ ]:
# Train the classifier on the patches and the corresponding features
body_overlap = 0.25 # Fraction overlap between the drawn annotations and the tiles
clf, patchesCDFsMod, annotationsMod = dianne.getClassifierForFromStrokes(drawings, patchCoordinates, tile_size, body_overlap, patch_size,
                                                                        ads, samples, qs, augFunc=dianne.PCMA, alpha=0.8, seed=0, showPatches=False)
dianne.saveGUIClassifier(clf, classifierPaths, 'some-name-gui', samples, ts, mpp, patch_size, tile_size, body_overlap, qs, patchesCDFsMod, annotationsMod, drawings)

# Run inference and visualize the results
if True:
    if not clf is None:
        infSample = samples[0]
        multiplier = 2 # Interpolation parameter for the probability heatmap
        x, y, p = dianne.inferProb(ads[infSample], clf, qs, tsize=ts/mpp, R=2, erode=False)
        xi, yi, pi = dianne.interpolatePoints(x, y, p, multiplier=multiplier)
        dianne.showProbImg(xi, yi, pi, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample, invert=False)
        viewer.set_overlay_points(xi, yi, pi, infSample, delta=tile_size / multiplier, alpha=0.5, color_low='#FFA500', color_high='#0000FF')

In [ ]:
# Create probability masks, extract contours and visualize them on the H&E image
downsampled_map, fshape = dianne.makeProbMask(ads[samples[0]], imgs[samples[0]], x, y, p, ts=ts, mpp=mpp, downfactor=16, verbose=True)
geojson = dianne.extractContoursForQuPath(downsampled_map, fshape, cutoff=0.8, min_area=10**6, downfactor=16, sigma=100)
dianne.viewContoursOnImage(imgs[samples[0]], geojson, fshape, level=2, figsize=(12, 12), linewidth=1)